In [ ]:
from IPython.display import HTML, display

organization_name = "DataVine Analytics"
group_name = "Group 15 Project"

projects = [
    "🍷 Wine Classification",
    "🐔 Agricultural Feed Recommendation",
    "🚔 Regional Crime Pattern Analysis"
]

team_members = [
    "Benedict Onyango",
    "Francis K. Mwangi",
    "Shem",
    "Sandra"
]

display(HTML(f"""
<div style="
    background: linear-gradient(135deg, #0F172A, #1E3A8A);
    color: white;
    padding: 30px;
    border-radius: 15px;
    text-align: center;
    font-family: Arial;
    box-shadow: 0px 4px 10px rgba(0,0,0,0.3);
">

<h1 style="font-size:42px;">📊 DataVine Analytics</h1>

<h2 style="font-size:28px;">Group 15 Projects</h2>

<h3 style="font-size:24px;">Summative Machine Learning Lab - Module 4</h3>

<h3 style="font-size:22px;">Projects</h3>

<div style="font-size:20px;">
🍷 Wine Classification<br>
🐔 Agricultural Feed Recommendation<br>
🚔 Regional Crime Pattern Analysis
</div>

<h3 style="font-size:22px;">Team Members</h3>

<div style="font-size:19px;">
1. Benedict Onyango<br>
2. Francis K. Mwangi<br>
3. Allan Abok<br>
4. Shem<br>
5. Sandra
</div>

<h3 style="font-size:20px;">Course: Data Science</h3>
<h3 style="font-size:20px;">Date: August 2026</h3>

</div>
"""))

### OVERVIEW

This notebook applies machine learning techniques to three datasets representing different business problems:

Wine Dataset — k-Nearest Neighbors classification with PCA and hyperparameter tuning.
Chickwts Dataset — recommendation system using PCA and cosine similarity.
USArrests Dataset — clustering using K-Means and Gaussian Mixture Models.

The objective is to follow a complete machine learning workflow involving data preparation, exploratory analysis, preprocessing, dimensionality reduction, model development, evaluation, visualization, and interpretation.

In [ ]:
# Crayons001
#Importing necesary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
print("Libraries imported successfully!")

# Part 1. Wine Classification System

## Business Objective

A premium wine distributor wants to automatically classify wine varieties using measurable chemical properties.

A k-Nearest Neighbors (k-NN) classification model will be developed. PCA will be used to reduce dimensionality while retaining 95% of the variance, and GridSearchCV will be used to identify the optimal k-NN hyperparameters.


In [ ]:
#Loading data
wine = pd.read_csv("wine.csv")
print("Wine dataset loaded successfully.")
print("Shape:", wine.shape)
wine.head()

Confirm:

Number of observations

Number of variables

Column names

Data types

Missing values

Duplicate records

In [ ]:
# Check data shape,column names , missing values and duplicate rows
print("Dataset shape:")
print(wine.shape)

print("\nColumn names:")
print(wine.columns.tolist())

print("\nData types:")
print(wine.dtypes)

print("\nMissing values:")
print(wine.isnull().sum())

print("\nDuplicate rows:")
print(wine.duplicated().sum())

In [ ]:
#Finding the target Column that contains Wine Variety/Class
wine.head()

In [ ]:
#To check Target
for column in wine.columns:
    print(column, ":", wine[column].nunique(), "unique values")

In [ ]:
#Check target distribution
target_column = "Wine"
print(wine[target_column].value_counts())

In [ ]:
#Visualize
plt.figure(figsize=(7, 5))
sns.countplot(
    data=wine,
    x=target_column
)
plt.title("Wine Class Distribution")
plt.xlabel("Wine Class")
plt.ylabel("Number of Samples")
plt.show()

In [ ]:
#Separate X and y
X = wine.drop(columns=["Wine"])
y = wine["Wine"]
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())

In [ ]:
#Encoding the target
#Converting categorical target labels into numeric values.originally it was 1,2,3 after encodning it becomes 0,1 and 2
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print("Original wine classes:")
print(label_encoder.classes_)
print("\nEncoded classes:")
print(np.unique(y_encoded))

In [ ]:
#Check Missing values 
print("Total missing values:", X.isnull().sum().sum())
print("Total duplicate rows:", wine.duplicated().sum())

In [ ]:
#Examining Numerical values 
#Proline has values in the hundreds/thousands, while some other variables have much smaller ranges.
#As such,standardization was applied  before PCA and k-NN.
X.describe().T

In [ ]:
#Split the data
# Creating training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

In [ ]:
#Creating Scaler to stardadize the data 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Standardization completed.")

In [ ]:
#Applying  PCA with 95% variance
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print("Original features:", X_train.shape[1])
print("PCA components:", X_train_pca.shape[1])

In [ ]:
#Checking the variance
print("Explained variance ratio:")
print(pca.explained_variance_ratio_)
print("\nTotal explained variance:")
print(pca.explained_variance_ratio_.sum())

In [ ]:
#Ploting the variance
plt.figure(figsize=(8, 5))
plt.plot(
    np.cumsum(pca.explained_variance_ratio_),
    marker="o"
)
plt.axhline(
    y=0.95,
    color="red",
    linestyle="--",
    label="95% variance"
)
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#Tuning the KNN-Model
knn = KNeighborsClassifier()
param_grid = {
    "n_neighbors": range(1, 21),
    "metric": ["euclidean", "manhattan", "minkowski"]
}
grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid_search.fit(X_train_pca, y_train)
print("Best parameters:")
print(grid_search.best_params_)
print("\nBest cross-validation accuracy:")
print(grid_search.best_score_)

In [ ]:
#Training the optimized model
best_knn = grid_search.best_estimator_
best_knn.fit(X_train_pca, y_train)
y_pred = best_knn.predict(X_test_pca)
print("Optimized k-NN model trained successfully.")

In [ ]:
#Calculating or getting the Model Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
#Classification Report
print(classification_report(
    y_test,
    y_pred,
    target_names=[str(c) for c in label_encoder.classes_]
))

In [ ]:
#Confusion Matrix-Visualizing Classification Results above
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title("Wine Classification Confusion Matrix")
plt.xlabel("Predicted Wine Class")
plt.ylabel("Actual Wine Class")
plt.show()

## Wine Classification Results & Interpretation
The Wine dataset contained 178 observations, 13 chemical features, and three wine classes. There were no missing values or duplicate observations.
The numerical features were standardized, and PCA reduced the 13 features to 10 components while retaining 95% of the variance.
GridSearchCV identified k = 18 and the Euclidean distance metric as the best k-NN parameters, achieving 97.91% cross-validation accuracy.
The optimized k-NN model achieved 100% test accuracy, correctly classifying all test observations with no misclassifications.
Overall, PCA combined with optimized k-NN provided highly accurate wine classification and could support automated wine identification, inventory management, and quality control.


## Part 2 — Chickwts Recommendation System

In [ ]:
#Loading the Data set
chickwts = pd.read_csv("chickwts.csv")
print("Shape:", chickwts.shape)
print(chickwts.columns)
chickwts.head()

In [ ]:
#Data inspection by information,missig values ,duplicates and summary
print(chickwts.info())
print("\nMissing values:")
print(chickwts.isnull().sum())
print("\nDuplicates:", chickwts.duplicated().sum())
print("\nSummary:")
print(chickwts.describe())

In [ ]:
#Check columns 
print(chickwts.columns.tolist())

In [ ]:
#Check types of feeds in the data and their count 
print(chickwts["feed"].value_counts())

In [ ]:
#Stardadize the weight
scaler_chick = StandardScaler()
chickwts["weight_scaled"] = scaler_chick.fit_transform(
    chickwts[["weight"]]
)
chickwts.head()

In [ ]:
#Applying PCA to one Component
pca_chick = PCA(n_components=1)
weight_pca = pca_chick.fit_transform(
    chickwts[["weight_scaled"]]
)
chickwts["PC1"] = weight_pca
print("Explained variance:", pca_chick.explained_variance_ratio_)

In [ ]:
#Calculating Cosine Similarity 
#Since each feed has multiple chicken observations,lets calculate the average PC1 value for each feed
feed_profiles = chickwts.groupby("feed")["PC1"].mean().to_frame()
print(feed_profiles)

In [ ]:
#Similarity
similarity_matrix = cosine_similarity(feed_profiles)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=feed_profiles.index,
    columns=feed_profiles.index
)
similarity_df

In [ ]:
#Creating a Recommendation Function 
def recommend_feed(feed_name, n=3):
    similarities = similarity_df[feed_name].sort_values(
        ascending=False
    ) 
    return similarities.drop(feed_name).head(n)

In [ ]:
#Testing it 
recommend_feed("casein")

## Chickwts Recommendation Results & Interpretation.

The Chickwts recommendation system standardized chicken weight, reduced it to one principal component using PCA, and calculated cosine similarity between feed types. For casein, the most similar feeds were meatmeal and sunflower, both with a similarity score of 1.00. Horsebean showed a similarity score of -1.00, indicating an opposite pattern.

## Part 3:USArrests clustering with K-Means and GMM

In [ ]:
#Loading the data set
USArrests = pd.read_csv("USArrests.csv")
print("Shape:", USArrests.shape)
print("Columns:", USArrests.columns.tolist())
USArrests.head()

In [ ]:
#Data Inspection Data information, Missing Values,Duplicates  and lastly data summary
print(USArrests.info())
print("\nMissing values:")
print(USArrests.isnull().sum())
print("\nDuplicates:", USArrests.duplicated().sum())
print("\nSummary:")
USArrests.describe()

In [ ]:
#Selecting Top 3 Features 
#The above referenced features are key and  directly measure crime activity, while UrbanPop represent population characteristics.
features = ["Murder", "Assault", "Rape"]
X_crime = USArrests[features]
print(X_crime.head())

In [ ]:
#Stardizing the features 
scaler_crime = StandardScaler()
X_crime_scaled = scaler_crime.fit_transform(X_crime)
print("Standardization completed.")

In [ ]:
#Apply PCA to 2 components
#The PCA has now reduced your 3 selected features to 2 components
pca_crime = PCA(n_components=2)
X_crime_pca = pca_crime.fit_transform(X_crime_scaled)
print("Explained variance ratio:")
print(pca_crime.explained_variance_ratio_)
print("\nTotal explained variance:")
print(pca_crime.explained_variance_ratio_.sum())

In [ ]:
#Elbow method to find optimal K for K-Means based on Inertia
inertias = []

K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    kmeans.fit(X_crime_pca)
    inertias.append(kmeans.inertia_)

In [ ]:
#Plotting the results
plt.figure(figsize=(8, 5))
plt.plot(
    K_range,
    inertias,
    marker="o"
)
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("K-Means Elbow Method")
plt.xticks(K_range)
plt.show()

In [ ]:
#From the grapph plotted above K appears to be 4, after 4 the curve decrease becomes gradual
#As such, Lets use K-Means with K=4
optimal_k = 4
kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)
kmeans_labels = kmeans.fit_predict(X_crime_pca)
USArrests["KMeans_Cluster"] = kmeans_labels
print(USArrests["KMeans_Cluster"].value_counts().sort_index())

In [ ]:
#Visulaize K-Means
plt.figure(figsize=(8, 6))
plt.scatter(
    X_crime_pca[:, 0],
    X_crime_pca[:, 1],
    c=kmeans_labels,
    cmap="viridis",
    s=60
)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clustering of USArrests")
plt.colorbar(label="Cluster")
plt.show()

In [ ]:
#Finding the best number of GMM clusters using BIC
bic_scores = []
for k in range(2, 11):
    gmm = GaussianMixture(
        n_components=k,
        random_state=42
    )  
    gmm.fit(X_crime_pca)
    bic_scores.append(gmm.bic(X_crime_pca))

In [ ]:
#Plotting BIC
plt.figure(figsize=(8, 5))
plt.plot(
    range(2, 11),
    bic_scores,
    marker="o"
)
plt.xlabel("Number of Clusters")
plt.ylabel("BIC")
plt.title("GMM BIC Scores")
plt.show()

In [ ]:
#K=2, BIC values is at its lowest at 2 (Bayesian Information Criterion)-Optimal number of Gaussian Components for the GMM Model
#Fitting the GMM Model
gmm = GaussianMixture(
    n_components=2,
    random_state=42
)
gmm_labels = gmm.fit_predict(X_crime_pca)
USArrests["GMM_Cluster"] = gmm_labels
print(USArrests["GMM_Cluster"].value_counts().sort_index())

In [ ]:
#Visulaize GMM Model
plt.figure(figsize=(8, 6))
plt.scatter(
    X_crime_pca[:, 0],
    X_crime_pca[:, 1],
    c=gmm_labels,
    cmap="plasma",
    s=60
)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("GMM Clustering of USArrests")
plt.colorbar(label="Cluster")
plt.show()

In [ ]:
#Comparing K-Means and GMM
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# K-Means
axes[0].scatter(
    X_crime_pca[:, 0],
    X_crime_pca[:, 1],
    c=kmeans_labels,
    cmap="viridis",
    s=50
)
axes[0].set_title("K-Means (K=4)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
# GMM
axes[1].scatter(
    X_crime_pca[:, 0],
    X_crime_pca[:, 1],
    c=gmm_labels,
    cmap="plasma",
    s=50
)
axes[1].set_title("GMM (K=2)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
plt.tight_layout()
plt.show()

## USArrests Clustering Results & Interpretation

The USArrests dataset was standardized using three crime-related features: Murder, Assault, and Rape. PCA reduced these features to two principal components.

The K-Means elbow method indicated that 4 clusters provided a suitable grouping. K-Means was then used to assign each observation to one of four clusters.

For the Gaussian Mixture Model, BIC indicated that 2 clusters provided the best fit. Unlike K-Means, GMM provides probabilistic cluster assignments.

The two methods produced different groupings, showing that the crime data can be interpreted using different clustering approaches. K-Means provides clear hard cluster assignments, while GMM offers a probabilistic approach that can better represent uncertainty between groups.

## OVERALL EVALUATION AND CONCLUSION 

The three machine learning projects successfully addressed different business problems.

Wine Classification: PCA reduced 13 features to 10 while retaining 95% variance. GridSearchCV selected k=18 with Euclidean distance, achieving 97.91% cross-validation accuracy and 100% test accuracy.
Feed Recommendation: PCA reduced the standardized weight data to one component. Cosine similarity identified feeds with similar performance patterns.
Crime Clustering: PCA reduced three crime features to two components. K-Means identified 4 clusters using the elbow method, while GMM identified 2 clusters based on the lowest BIC.

Overall, the analysis demonstrates how classification, recommendation, PCA, and clustering techniques can provide useful insights for business decision-making.